# Gray-Box AIA72 — Fallback Failure / Dual-Outage Stress Test

Post-hoc robustness analysis of the frozen v3 architecture. This notebook quantifies when **ABSTAIN is unavoidable** because SHARP—the fallback path—fails. No AIA-only rescue route is introduced.

In [1]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib, json
import numpy as np
import pandas as pd

# ============================================================
# Gray-Box AIA72 — Post-hoc Fallback Failure / Dual-Outage Stress Test
# ============================================================
#
# SCIENTIFIC STATUS:
# - post-hoc operational robustness analysis
# - no model fitting/retraining
# - no threshold tuning
# - no policy reselection
# - no policy redesign
#
# PURPOSE:
# Quantify when the ABSTAIN state becomes unavoidable.
#
# Frozen v3 semantics:
# NORMAL   = trusted SHARP+AIA fusion
# DEGRADED = SHARP fallback when fusion/AIA is not usable
# ABSTAIN  = SHARP fallback itself is unavailable
#
# Because the frozen fusion branch depends on SHARP + AIA,
# a SHARP outage removes both the fusion path and the fallback path.
# Therefore this notebook does NOT invent an AIA-only rescue route.
#
# Scenarios:
# 1) Random SHARP outage: 1%, 5%, 10%, 20%, 40%
# 2) Random dual outage: simultaneous AIA+SHARP loss at 1%, 5%, 10%, 20%
# 3) Independent AIA + SHARP missingness combinations
# 4) Contiguous SHARP outages: ~6h, 12h, 24h, 48h
# 5) Contiguous dual outages: ~6h, 12h, 24h, 48h

EVAL_DIR = Path("/home/abmoses2000/graybox_aia72_supplementary2026_v3_eval_20261005")
CASES = EVAL_DIR / "supplementary2026_policy_cases.csv.gz"
SUMMARY = EVAL_DIR / "summary.json"

V3_DIR = Path("/home/abmoses2000/graybox_aia72_policy_v3_fallback_first_freeze_20261005")
V3_PROTOCOL = V3_DIR / "protocol.json"

ALARM_DIR = Path("/home/abmoses2000/graybox_aia72_alarm_and_primary_policy_freeze_v1_20261005")
ALARM_THRESHOLDS = ALARM_DIR / "alarm_thresholds.csv"

CUTS = Path("/home/abmoses2000/graybox_aia72_policy_freeze_v1_20261005/candidate_cutoffs.csv")

OUT = Path("/home/abmoses2000/graybox_aia72_posthoc_fallback_failure_v1_20261005")
if OUT.exists():
    raise SystemExit(f"Refusing to overwrite existing output: {OUT}")
OUT.mkdir(parents=True)

PRIMARY_PCT = 0.90
SEED = 20261005
NREP = 100

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda:f.read(4*1024*1024),b""):
            h.update(chunk)
    return h.hexdigest()

def require(cond,msg):
    if not cond:
        raise ValueError(msg)

def binary_metrics(y,pred):
    y=np.asarray(y,int); pred=np.asarray(pred,int)
    tp=int(((y==1)&(pred==1)).sum())
    fp=int(((y==0)&(pred==1)).sum())
    fn=int(((y==1)&(pred==0)).sum())
    tn=int(((y==0)&(pred==0)).sum())
    recall=tp/(tp+fn) if tp+fn else np.nan
    spec=tn/(tn+fp) if tn+fp else np.nan
    prec=tp/(tp+fp) if tp+fp else np.nan
    f1=2*prec*recall/(prec+recall) if np.isfinite(prec) and np.isfinite(recall) and (prec+recall)>0 else np.nan
    tss=recall+spec-1 if np.isfinite(recall) and np.isfinite(spec) else np.nan
    denom=((tp+fn)*(fn+tn)+(tp+fp)*(fp+tn))
    hss=2*(tp*tn-fn*fp)/denom if denom else np.nan
    return dict(tp=tp,fp=fp,fn=fn,tn=tn,recall=float(recall),specificity=float(spec),
                precision=float(prec),f1=float(f1),tss=float(tss),hss=float(hss))

for p in [CASES,SUMMARY,V3_PROTOCOL,ALARM_THRESHOLDS,CUTS]:
    require(p.is_file(),f"Missing input: {p}")

summary=json.loads(SUMMARY.read_text())
v3_protocol=json.loads(V3_PROTOCOL.read_text())

require(summary["status"]=="SUPPLEMENTARY_2026_FALLBACK_V3_CLEAN_EVALUATION_COMPLETE",
        "Unexpected evaluation source")
require(summary["policy_reselection_performed"] is False,"Source reselection detected")
require(summary["threshold_tuning_performed"] is False,"Source tuning detected")
require(v3_protocol["primary_policy_id"]=="fallback_v3_q90","Primary v3 changed")

df=pd.read_csv(CASES,low_memory=False)
alarm=pd.read_csv(ALARM_THRESHOLDS)
cutoffs=pd.read_csv(CUTS)
alarm_map={r.branch:float(r.threshold) for r in alarm.itertuples()}

def cutoff(metric,pct):
    hit=cutoffs.loc[(cutoffs["metric"]==metric)&np.isclose(cutoffs["percentile"].astype(float),pct)]
    require(len(hit)==1,f"Missing cutoff {metric} q{pct}")
    return float(hit.iloc[0]["cutoff"])

required=[
    "label","issue_utc","sharp_probability","fusion_probability",
    "fusion_seed_std","sharp_aia_probability_gap","fusion_entropy",
    "sharp_aia_mondrian_state"
]
missing=[c for c in required if c not in df.columns]
require(not missing,f"Missing columns: {missing}")

y=df["label"].astype(int).to_numpy()
sharp_p=df["sharp_probability"].astype(float).to_numpy()
fusion_p=df["fusion_probability"].astype(float).to_numpy()

clean_sharp_ok=np.isfinite(sharp_p)
clean_aia_ok=np.isfinite(fusion_p)

fusion_single=np.isin(
    df["sharp_aia_mondrian_state"].astype(str).to_numpy(),
    ["singleton_no_flare","singleton_flare"]
)

fs=cutoff("fusion_seed_std",PRIMARY_PCT)
gap=cutoff("sharp_aia_probability_gap",PRIMARY_PCT)
ent=cutoff("fusion_entropy",PRIMARY_PCT)

frozen_fusion_trust=(
    fusion_single &
    (df["fusion_seed_std"].to_numpy(float)<=fs) &
    (df["sharp_aia_probability_gap"].to_numpy(float)<=gap) &
    (df["fusion_entropy"].to_numpy(float)<=ent)
)

sharp_pred=(sharp_p>=alarm_map["sharp"]).astype(int)
fusion_pred=(fusion_p>=alarm_map["sharp_aia"]).astype(int)

issue_dt=pd.to_datetime(df["issue_utc"],utc=True,errors="coerce")
require(issue_dt.notna().all(),"issue_utc parse failure")

def evaluate(name,aia_ok_mask,sharp_ok_mask,scenario_type,aia_param,sharp_param,replicate):
    aia_ok=np.asarray(aia_ok_mask,bool)
    sharp_ok=np.asarray(sharp_ok_mask,bool)

    # Frozen architecture: fusion requires both modalities.
    fusion_ok=aia_ok & sharp_ok

    # Baseline fusion
    fusion_issue=fusion_ok
    f_pred=np.full(len(y),-1,dtype=int)
    f_pred[fusion_issue]=fusion_pred[fusion_issue]

    # Baseline SHARP
    sharp_issue=sharp_ok
    s_pred=np.full(len(y),-1,dtype=int)
    s_pred[sharp_issue]=sharp_pred[sharp_issue]

    # Frozen v3
    normal=fusion_ok & frozen_fusion_trust
    degraded=(~normal) & sharp_ok
    abstain=~(normal|degraded)

    v_pred=np.full(len(y),-1,dtype=int)
    v_pred[normal]=fusion_pred[normal]
    v_pred[degraded]=sharp_pred[degraded]

    rows=[]
    for system,issued,pred,norm,deg,absn in [
        ("baseline_always_fusion",fusion_issue,f_pred,fusion_issue,np.zeros(len(y),bool),~fusion_issue),
        ("baseline_always_sharp",sharp_issue,s_pred,np.zeros(len(y),bool),sharp_issue,~sharp_issue),
        ("fallback_v3_q90",normal|degraded,v_pred,normal,degraded,abstain),
    ]:
        m=binary_metrics(y[issued],pred[issued]) if issued.any() else {
            k:np.nan for k in ["tp","fp","fn","tn","recall","specificity","precision","f1","tss","hss"]
        }
        positives=int((y==1).sum())
        rows.append({
            "scenario":name,
            "scenario_type":scenario_type,
            "aia_parameter":aia_param,
            "sharp_parameter":sharp_param,
            "replicate":replicate,
            "system":system,
            "requested_cases":len(y),
            "positives":positives,
            "normal_cases":int(norm.sum()),
            "degraded_cases":int(deg.sum()),
            "abstain_cases":int(absn.sum()),
            "availability_rate":float(issued.mean()),
            "positive_retention_rate":float(((y==1)&issued).sum()/positives),
            "abstained_flares":int(((y==1)&absn).sum()),
            "false_alarms":int(((y==0)&issued&(pred==1)).sum()),
            **{f"issued_{k}":v for k,v in m.items()}
        })
    return rows

rows=[]

# Clean reference
rows += evaluate(
    "clean_reference",
    np.ones(len(df),bool),
    np.ones(len(df),bool),
    "reference",
    0.0,0.0,0
)

# 1) Random SHARP outages, AIA otherwise healthy
rng=np.random.default_rng(SEED)
for sr in [0.01,0.05,0.10,0.20,0.40]:
    for rep in range(NREP):
        sharp_ok=rng.random(len(df))>=sr
        aia_ok=np.ones(len(df),bool)
        rows += evaluate(
            f"random_sharp_outage_{int(sr*100)}pct",
            aia_ok,sharp_ok,
            "random_sharp_outage",
            0.0,sr,rep
        )

# 2) Simultaneous dual outages
rng=np.random.default_rng(SEED+1)
for dr in [0.01,0.05,0.10,0.20]:
    for rep in range(NREP):
        healthy=rng.random(len(df))>=dr
        rows += evaluate(
            f"random_dual_outage_{int(dr*100)}pct",
            healthy,healthy,
            "random_dual_outage",
            dr,dr,rep
        )

# 3) Independent AIA and SHARP losses
rng=np.random.default_rng(SEED+2)
for ar,sr in [(0.10,0.01),(0.10,0.05),(0.20,0.05),(0.20,0.10),(0.40,0.10),(0.40,0.20)]:
    for rep in range(NREP):
        aia_ok=rng.random(len(df))>=ar
        sharp_ok=rng.random(len(df))>=sr
        rows += evaluate(
            f"independent_aia{int(ar*100)}_sharp{int(sr*100)}",
            aia_ok,sharp_ok,
            "independent_missingness",
            ar,sr,rep
        )

# 4/5) Contiguous SHARP-only and dual outages
unique_times=np.array(sorted(issue_dt.unique()))
cadence_minutes=96
hours_to_steps={6:max(1,round(6*60/cadence_minutes)),
                12:max(1,round(12*60/cadence_minutes)),
                24:max(1,round(24*60/cadence_minutes)),
                48:max(1,round(48*60/cadence_minutes))}

for hours,steps in hours_to_steps.items():
    if steps>=len(unique_times):
        continue
    starts=np.linspace(0,len(unique_times)-steps-1,num=min(50,len(unique_times)-steps),dtype=int)
    starts=np.unique(starts)

    for rep,start in enumerate(starts):
        bad=set(unique_times[start:start+steps])
        sharp_ok=~issue_dt.isin(bad).to_numpy()

        rows += evaluate(
            f"contiguous_sharp_outage_{hours}h",
            np.ones(len(df),bool),sharp_ok,
            "contiguous_sharp_outage",
            0.0,float(hours),rep
        )

        rows += evaluate(
            f"contiguous_dual_outage_{hours}h",
            sharp_ok,sharp_ok,
            "contiguous_dual_outage",
            float(hours),float(hours),rep
        )

res=pd.DataFrame(rows)

metrics=[
    "availability_rate","positive_retention_rate","abstained_flares",
    "false_alarms","issued_recall","issued_specificity","issued_precision",
    "issued_f1","issued_tss","issued_hss"
]

agg_rows=[]
for (stype,ap,sp,system),g in res.groupby(
    ["scenario_type","aia_parameter","sharp_parameter","system"],dropna=False
):
    row={
        "scenario_type":stype,
        "aia_parameter":ap,
        "sharp_parameter":sp,
        "system":system,
        "n_replicates":len(g)
    }
    for m in metrics:
        vals=pd.to_numeric(g[m],errors="coerce").to_numpy(float)
        row[f"{m}_mean"]=float(np.nanmean(vals))
        row[f"{m}_p05"]=float(np.nanpercentile(vals,5))
        row[f"{m}_p95"]=float(np.nanpercentile(vals,95))
    agg_rows.append(row)

agg=pd.DataFrame(agg_rows)

# v3 vs fusion and v3 vs SHARP operational deltas
key=["scenario","scenario_type","aia_parameter","sharp_parameter","replicate"]
v3=res.loc[res["system"]=="fallback_v3_q90"]
fus=res.loc[res["system"]=="baseline_always_fusion"]
shp=res.loc[res["system"]=="baseline_always_sharp"]

pf=v3.merge(fus,on=key,suffixes=("_v3","_fusion"))
ps=v3.merge(shp,on=key,suffixes=("_v3","_sharp"))

delta_f=pd.DataFrame({
    **{k:pf[k] for k in key},
    "availability_gain_v3_vs_fusion":pf["availability_rate_v3"]-pf["availability_rate_fusion"],
    "flare_retention_gain_v3_vs_fusion":pf["positive_retention_rate_v3"]-pf["positive_retention_rate_fusion"],
    "abstained_flare_reduction_v3_vs_fusion":pf["abstained_flares_fusion"]-pf["abstained_flares_v3"],
    "tss_delta_v3_vs_fusion":pf["issued_tss_v3"]-pf["issued_tss_fusion"],
    "false_alarm_delta_v3_vs_fusion":pf["false_alarms_v3"]-pf["false_alarms_fusion"],
})

delta_s=pd.DataFrame({
    **{k:ps[k] for k in key},
    "availability_gain_v3_vs_sharp":ps["availability_rate_v3"]-ps["availability_rate_sharp"],
    "flare_retention_gain_v3_vs_sharp":ps["positive_retention_rate_v3"]-ps["positive_retention_rate_sharp"],
    "abstained_flare_delta_v3_vs_sharp":ps["abstained_flares_v3"]-ps["abstained_flares_sharp"],
    "tss_delta_v3_vs_sharp":ps["issued_tss_v3"]-ps["issued_tss_sharp"],
    "false_alarm_delta_v3_vs_sharp":ps["false_alarms_v3"]-ps["false_alarms_sharp"],
})

res_path=OUT/"scenario_level_results.csv.gz"
agg_path=OUT/"scenario_summary.csv"
df_path=OUT/"deltas_v3_vs_fusion.csv"
ds_path=OUT/"deltas_v3_vs_sharp.csv"
protocol_path=OUT/"protocol.json"

res.to_csv(res_path,index=False,compression={"method":"gzip","mtime":0})
agg.to_csv(agg_path,index=False)
delta_f.to_csv(df_path,index=False)
delta_s.to_csv(ds_path,index=False)

protocol={
    "status":"POSTHOC_FALLBACK_FAILURE_V1_COMPLETE",
    "scientific_acceptance":False,
    "created_utc":datetime.now(timezone.utc).isoformat(),
    "analysis_status":"posthoc_robustness_not_independent_validation",
    "primary_policy_id":"fallback_v3_q90",
    "policy_reselection_performed":False,
    "threshold_tuning_performed":False,
    "policy_redesign_performed":False,
    "architecture_fact":"Frozen fusion requires both SHARP and AIA; no AIA-only rescue path was invented post hoc.",
    "abstain_interpretation":"ABSTAIN becomes unavoidable whenever SHARP fallback is unavailable.",
    "scenarios":{
        "random_sharp_outage_rates":[0.01,0.05,0.10,0.20,0.40],
        "random_dual_outage_rates":[0.01,0.05,0.10,0.20],
        "independent_missingness_pairs":[
            [0.10,0.01],[0.10,0.05],[0.20,0.05],
            [0.20,0.10],[0.40,0.10],[0.40,0.20]
        ],
        "contiguous_sharp_outage_hours":[6,12,24,48],
        "contiguous_dual_outage_hours":[6,12,24,48]
    },
    "source_sha256":{
        "supplementary2026_policy_cases":sha256(CASES),
        "supplementary2026_summary":sha256(SUMMARY),
        "v3_protocol":sha256(V3_PROTOCOL),
        "alarm_thresholds":sha256(ALARM_THRESHOLDS),
        "candidate_cutoffs":sha256(CUTS),
    },
    "output_sha256":{
        res_path.name:sha256(res_path),
        agg_path.name:sha256(agg_path),
        df_path.name:sha256(df_path),
        ds_path.name:sha256(ds_path),
    }
}
protocol_path.write_text(json.dumps(protocol,indent=2)+"\n")

print("===== POST-HOC FALLBACK FAILURE / DUAL-OUTAGE V1 COMPLETE =====")
print("policy reselection performed: False")
print("threshold tuning performed: False")
print("policy redesign performed: False")
print("status: post-hoc robustness, NOT independent validation")
print("ABSTAIN interpretation: unavoidable when SHARP fallback is unavailable")

pd.set_option("display.max_rows",300)
print("\nScenario summary:")
print(agg.to_string(index=False))

print("\nV3 vs fusion grouped mean deltas:")
print(
    delta_f.groupby(["scenario_type","aia_parameter","sharp_parameter"],dropna=False)
    .agg({
        "availability_gain_v3_vs_fusion":"mean",
        "flare_retention_gain_v3_vs_fusion":"mean",
        "abstained_flare_reduction_v3_vs_fusion":"mean",
        "tss_delta_v3_vs_fusion":"mean",
        "false_alarm_delta_v3_vs_fusion":"mean",
    }).reset_index().to_string(index=False)
)

print("\nSaved:",protocol_path)
print("Saved:",res_path)
print("Saved:",agg_path)
print("Saved:",df_path)
print("Saved:",ds_path)


===== POST-HOC FALLBACK FAILURE / DUAL-OUTAGE V1 COMPLETE =====
policy reselection performed: False
threshold tuning performed: False
policy redesign performed: False
status: post-hoc robustness, NOT independent validation
ABSTAIN interpretation: unavoidable when SHARP fallback is unavailable

Scenario summary:
          scenario_type  aia_parameter  sharp_parameter                 system  n_replicates  availability_rate_mean  availability_rate_p05  availability_rate_p95  positive_retention_rate_mean  positive_retention_rate_p05  positive_retention_rate_p95  abstained_flares_mean  abstained_flares_p05  abstained_flares_p95  false_alarms_mean  false_alarms_p05  false_alarms_p95  issued_recall_mean  issued_recall_p05  issued_recall_p95  issued_specificity_mean  issued_specificity_p05  issued_specificity_p95  issued_precision_mean  issued_precision_p05  issued_precision_p95  issued_f1_mean  issued_f1_p05  issued_f1_p95  issued_tss_mean  issued_tss_p05  issued_tss_p95  issued_hss_mean  iss